# TTD Dataset Demo

This notebook follows the modality guidance called out in issue #126 and demonstrates:
- Direct SQL
- REST API
- MCP tools
- Agentic access through the orchestrator

In [ ]:
import json
import os
import sqlite3
from pathlib import Path

import requests

REQUEST_TIMEOUT = int(os.getenv("TTD_NOTEBOOK_TIMEOUT", "20"))
TTD_NODE_URL = os.getenv("TTD_NODE_URL", "http://ca-ttd-agentic-prod").rstrip("/")
ORCHESTRATOR_URL = os.getenv(
    "ORCHESTRATOR_URL",
    "https://ca-orchestrator-agentic-prod.bluesmoke-f9a71a0e.westus3.azurecontainerapps.io",
).rstrip("/")
TTD_DB_PATH = Path(os.getenv("TTD_DB_PATH", "/app/data/ttd.sqlite3"))

print("TTD_NODE_URL:", TTD_NODE_URL)
print("ORCHESTRATOR_URL:", ORCHESTRATOR_URL)
print("TTD_DB_PATH:", TTD_DB_PATH)

## Section A: Setup and environment preflight
Validate endpoint reachability before running modality-specific checks.

In [ ]:
checks = [
    ("ttd-health", f"{TTD_NODE_URL}/health"),
    ("ttd-summary", f"{TTD_NODE_URL}/data/summary"),
    ("orchestrator-health", f"{ORCHESTRATOR_URL}/health"),
]

for name, url in checks:
    try:
        resp = requests.get(url, timeout=REQUEST_TIMEOUT)
        print(f"{name:20s} -> {resp.status_code}")
    except Exception as exc:
        print(f"{name:20s} -> ERROR: {exc}")

## Section B: Direct SQL
This section demonstrates SQL-level access in two ways:
1. Direct local SQLite file access (if DB file is available in notebook runtime)
2. SQL submitted through the node `/data/query` route (read-only guard enforced by node)

In [ ]:
direct_sql_query = """
SELECT c.name, t.target_name, i.metric_type, i.metric_value, i.metric_unit
FROM interactions i
JOIN compounds c ON c.id = i.compound_id
JOIN targets t ON t.id = i.target_id
LIMIT 10
"""

print("=== B1) Local SQLite direct SQL ===")
if TTD_DB_PATH.exists():
    with sqlite3.connect(TTD_DB_PATH) as conn:
        rows = conn.execute(direct_sql_query).fetchall()
    print(f"rows={len(rows)}")
    for row in rows[:5]:
        print(row)
else:
    print(f"Local DB not found at {TTD_DB_PATH}; skipping direct-file SQL check.")

print("\n=== B2) SQL via /data/query (read-only SQL over REST) ===")
try:
    resp = requests.post(
        f"{TTD_NODE_URL}/data/query",
        json={"sql": direct_sql_query},
        timeout=REQUEST_TIMEOUT,
    )
    print("status:", resp.status_code)
    payload = resp.json()
    print("keys:", list(payload.keys()))
    preview = payload.get("rows", [])[:3]
    print("preview rows:", json.dumps(preview, indent=2) if preview else preview)
except Exception as exc:
    print("query error:", exc)

## Section C: REST API
Call dataset-specific and generic data routes directly on the TTD node.

In [ ]:
calls = [
    ("summary", "GET", f"{TTD_NODE_URL}/data/summary", None),
    ("tables", "GET", f"{TTD_NODE_URL}/data/tables", None),
    ("ttd-search-egfr", "GET", f"{TTD_NODE_URL}/ttd/search", {"q": "EGFR", "limit": 5}),
    ("ttd-target-p00533", "GET", f"{TTD_NODE_URL}/ttd/target/P00533", None),
]

for label, method, url, params in calls:
    try:
        if method == "GET":
            r = requests.get(url, params=params, timeout=REQUEST_TIMEOUT)
        else:
            r = requests.request(method, url, json=params, timeout=REQUEST_TIMEOUT)
        print(f"\n{label} -> {r.status_code}")
        body = r.json()
        print(json.dumps(body, indent=2)[:1200])
    except Exception as exc:
        print(f"{label} -> ERROR: {exc}")

## Section D: MCP Tools
Discover MCP tools from the TTD node and invoke a tool call.

In [ ]:
mcp_tools = []
try:
    r = requests.get(f"{TTD_NODE_URL}/mcp/tools", timeout=REQUEST_TIMEOUT)
    print("/mcp/tools status:", r.status_code)
    data = r.json()
    mcp_tools = data if isinstance(data, list) else data.get("tools", [])
    print("tool_count:", len(mcp_tools))
    for t in mcp_tools[:10]:
        name = t.get("name") if isinstance(t, dict) else None
        desc = t.get("description") if isinstance(t, dict) else None
        print("-", name, "::", (desc or "")[:120])
except Exception as exc:
    print("mcp tools discovery error:", exc)

In [ ]:
candidate_names = []
for t in mcp_tools:
    if not isinstance(t, dict):
        continue
    nm = (t.get("name") or "").strip()
    if nm:
        candidate_names.append(nm)

search_tool = next((n for n in candidate_names if "search" in n.lower()), None)
target_tool = next((n for n in candidate_names if "target" in n.lower()), None)
tool_name = search_tool or target_tool

print("selected_tool:", tool_name)
if not tool_name:
    print("No usable MCP tool discovered in /mcp/tools output.")
else:
    payload_new = {
        "name": tool_name,
        "arguments": {"q": "EGFR", "limit": 5} if "search" in tool_name.lower() else {"uniprot_id": "P00533"},
    }
    payload_legacy = {
        "tool": tool_name,
        "args": {"q": "EGFR", "limit": 5} if "search" in tool_name.lower() else {"uniprot_id": "P00533"},
    }
    try:
        resp = requests.post(f"{TTD_NODE_URL}/mcp/call", json=payload_new, timeout=REQUEST_TIMEOUT)
        if resp.status_code >= 400:
            resp = requests.post(f"{TTD_NODE_URL}/mcp/call", json=payload_legacy, timeout=REQUEST_TIMEOUT)
        print("/mcp/call status:", resp.status_code)
        print(resp.text[:2000])
    except Exception as exc:
        print("mcp call error:", exc)

## Section E: Agentic Access (Orchestrator)
Use orchestrator `/chat/stream` so routing and delegation behavior can be observed end-to-end.

In [ ]:
query = "Find high-affinity EGFR interactions from the TTD dataset and summarize top hits."
payload = {"message": query, "agent": "orchestrator"}

print("sending to:", f"{ORCHESTRATOR_URL}/chat/stream")
print("query:", query)

try:
    with requests.post(
        f"{ORCHESTRATOR_URL}/chat/stream",
        json=payload,
        timeout=REQUEST_TIMEOUT,
        stream=True,
    ) as resp:
        print("status:", resp.status_code)
        resp.raise_for_status()
        chunks = []
        for line in resp.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data:"):
                continue
            raw = line[5:].strip()
            if not raw:
                continue
            try:
                evt = json.loads(raw)
            except json.JSONDecodeError:
                continue
            if evt.get("type") == "token":
                chunks.append(evt.get("content", ""))
            elif evt.get("type") == "error":
                print("stream error:", evt)
                break
            elif evt.get("type") == "done":
                break

        answer = "".join(chunks).strip()
        print("\nassistant response:\n")
        print(answer if answer else "(no tokens returned)")
except Exception as exc:
    print("agentic call failed:", exc)